In [1]:
import numpy as np
from tqdm import tqdm

The neural network we are going to recreate is made up of only one neuron. Basically, it is just a perceptron and we are going to try to optimize that.

An Important insight is that these neural networks cannot really generate negative values here. It is a leaky abstraction. Because of the fact that we are using relu activation we are unable to generate a negative value. Because it just clips it down to a negative value. And, also the gradient is such that if the activations are negative. Then, the gradient should be zero.

In [2]:
X = np.array([[1, 3, 1], [2, 2, 2], [3, 5, 3], [4, 1, 4], [5, 1, 5], [-1, 3, 6], [-2, 10, 7], [1, -4, 8], [2, 5, 9], [0, 3, 10], [1, 0, 11], [-2, 20, 12]])
Y = np.array([11, 13, 19, 18, 21, -2, 0, -3, 9, -2, -1, 5])

In [1055]:
weight = np.array([1, 1, 1])
bias = np.array(1)

In [1056]:
N = len(X)
learning_rate = 0.001

In [1060]:
for i in tqdm(range(100000)):
    activations = np.maximum(0, X @ weight + bias)
    error = activations - Y

    error[activations <= 0] = 0


    gradient_w = 2/N *  np.sum( error * X.T , axis=1)
    gradient_b = 2/N *  np.sum(error)





    weight = weight - learning_rate*gradient_w
    bias = bias - learning_rate*gradient_b

    # if i % 100 == 0:
    # print(f"Weight = {weight} Bias = {bias}")

100%|██████████| 100000/100000 [00:02<00:00, 36883.38it/s]


In [1061]:
print(f"Weight = {weight} Bias = {bias}")

Weight = [ 4.00000012  1.00000004 -1.00000001] Bias = 4.9999995530323265


Another important insight is that if it jumps too far in the negative, it is hard for it to recover because the gradient then just collapses immmediately.

In [13]:
X = np.array([[1, 3, 1], [2, 2, 2], [3, 5, 3], [4, 1, 4], [5, 1, 5], [-1, 3, 6], [-2, 10, 7], [1, -4, 8], [2, 5, 9], [0, 3, 10], [1, 0, 11], [-2, 20, 12]])
Y = np.array([[123], [129], [243], [168], [201], [169], [357], [74], [322], [250], [202], [707]])


In [22]:
# W1 = np.array([[2, 1, 5], [1, 4, 0], [3, 2, 1], [1, 3, 2]])
# B1 = np.array([1, 1, 0, 1])

# W2 = np.array([[1, 2, 3, 4]])
# B2 = np.array([2])
# W1.shape, W2.shape, B1.shape, B2.shape

In [23]:
# for i in range(12):
#     f0 = W1 @ X[i] + B1
#     h1 = np.maximum(f0, 0)
#     print(W2 @ h1 + B2)


In [14]:
X = X.reshape(12, 3, 1)
Y = Y.reshape(12, 1, 1)

In [64]:
class NeuralNetwork():

    def __init__(self, X, Y):

        # np.random.randint(1, 5, size=(4, 3))

        W0 = np.random.uniform(0, 0.3, size=(1, 100, 3))

        B0 = np.random.uniform(0, 0.3, size=(1, 100, 1))

        W1 = np.random.uniform(0, 0.3, size=(1, 100, 100))

        B1 = np.random.uniform(0, 0.3, size=(1, 100, 1))

        W2 = np.random.uniform(0, 0.3, size=(1, 1, 100))

        B2 = np.random.uniform(0, 0.3, size=(1, 1, 1))

        # W0 = np.array([[2, 1, 5], [1, 4, 0], [3, 2, 1], [1, 3, 2]])
        # B0 = np.array([[1],
        #                [1],
        #                [0],
        #                [1]])

        # W1 = np.array([[1, 2, 3, 4]])
        # B1 = np.array([[2]])

        self.W = [W0, W1, W2]

        self.B = [B0, B1, B2]

        self.X = X

        self.Y = Y

        self.f = []

        self.h = []

        self.gradients_f = []

        self.gradients_W = []

        self.gradients_B = []

    def forward_pass(self):

        f0 = self.W[0] @ self.X + self.B[0]

        h1 = np.maximum(f0, 0)

        f1 = self.W[1] @ h1 + self.B[1]

        h2 = np.maximum(f1, 0)

        f2 = self.W[2] @ h2 + self.B[2]

        self.f = [f0, f1, f2]

        self.h = [0, h1, h2]

    def get_gradients(self):


        diagonal_matrices_1 = []
        for i in range(12):
            diagonal_matrix_1 = np.diag(self.f[1][i].flatten())
            np.fill_diagonal(diagonal_matrix_1, np.maximum(diagonal_matrix_1.diagonal(), 0))

            diagonal_matrices_1.append(diagonal_matrix_1)

        diagonal_matrices_1 = np.array(diagonal_matrices_1)

        diagonal_matrices_0 = []
        for i in range(12):
            diagonal_matrix_0 = np.diag(self.f[0][i].flatten())
            np.fill_diagonal(diagonal_matrix_0, np.maximum(diagonal_matrix_0.diagonal(), 0))

            diagonal_matrices_0.append(diagonal_matrix_0)

        diagonal_matrices_0 = np.array(diagonal_matrices_0)


        gradient_f2 = 2 * (self.f[2] - self.Y)
        gradient_f1 = gradient_f2 @ self.W[2] @ diagonal_matrices_1
        gradient_f0 = gradient_f1 @ self.W[1] @ diagonal_matrices_0


        gradient_B2 = np.mean( np.transpose(gradient_f2, (0, 2, 1)), axis=0)[np.newaxis, :, :]
        gradient_B1 = np.mean( np.transpose(gradient_f1, (0, 2, 1)), axis=0)[np.newaxis, :, :]
        gradient_B0 = np.mean( np.transpose(gradient_f0, (0, 2, 1)), axis=0)[np.newaxis, :, :]



        gradient_W2 = np.mean( np.transpose(self.h[2] @ gradient_f2, (0, 2, 1)), axis=0 )[np.newaxis, :, :]
        gradient_W1 = np.mean( np.transpose(self.h[1] @ gradient_f1, (0, 2, 1)), axis=0 )[np.newaxis, :, :]
        gradient_W0 = np.mean( np.transpose(self.X @ gradient_f0, (0, 2, 1)), axis=0 )[np.newaxis, :, :]




        self.gradients_f = [gradient_f0, gradient_f1]
        self.gradients_B = [gradient_B0, gradient_B1, gradient_B2]
        self.gradients_W = [gradient_W0, gradient_W1, gradient_W2]





        # diagonal_matrix_0 = np.diag(self.f[0].flatten())
        # np.fill_diagonal(diagonal_matrix_0, np.maximum(diagonal_matrix_0.diagonal(), 0))

        # diagonal_matrix_1 = np.diag(self.f[1].flatten())
        # np.fill_diagonal(diagonal_matrix_1, np.maximum(diagonal_matrix_1.diagonal(), 0))

        # gradient_f2 = 2 * (self.f[2] - y)
        # gradient_f1 = gradient_f2 @ self.W[2] @ diagonal_matrix_1

        # gradient_f0 = gradient_f1 @ self.W[1] @ diagonal_matrix_0

        # gradient_B0 = gradient_f0.T
        # gradient_B1 = gradient_f1.T
        # gradient_B2 = gradient_f2.T

        # gradient_W0 = (x @ gradient_f0).T
        # gradient_W1 = (self.h[1] @ gradient_f1).T
        # gradient_W2 = (self.h[2] @ gradient_f2).T

        # self.gradient_f.append( [gradient_f0, gradient_f1, gradient_f2] )
        # self.gradient_W.append( [gradient_W0, gradient_W1, gradient_W2] )
        # self.gradient_B.append( [gradient_B0, gradient_B1, gradient_B2] )

    def gradient_descent(self, learning_rate= 0.00000001):

        for i in range(3):
            self.B[i] = self.B[i] - learning_rate*self.gradients_B[i]
            self.W[i] = self.W[i] - learning_rate*self.gradients_W[i]

        # self.B[0] = self.B[0] - learning_rate*self.gradients_B[0]
        # self.B[1] = self.B[1] - learning_rate*self.gradients_B[1]
        # self.B[2] = self.B[2] - learning_rate*self.gradients_B[2]

        # self.W[0] = self.W[0] - learning_rate*self.gradients_W[0]
        # self.W[1] = self.W[1] - learning_rate*self.gradients_W[1]
        # self.W[2] = self.W[2] - learning_rate*self.gradients_W[2]

    def calculate_mse(self):

        self.forward_pass(self.X)

        return np.mean((self.Y - self.f[2]) ** 2)

    def train(self, epochs = 100):

        for _ in tqdm(range(epochs)):


            self.forward_pass()
            self.get_gradients()
            self.gradient_descent()

In [65]:
nn = NeuralNetwork(X, Y)


In [66]:
nn.forward_pass()

In [67]:
nn.train(epochs=10000)

  0%|          | 0/10000 [00:00<?, ?it/s]

100%|██████████| 10000/10000 [02:34<00:00, 64.62it/s]


In [69]:
nn.calculate_mse()

TypeError: NeuralNetwork.forward_pass() takes 1 positional argument but 2 were given

In [94]:
outputs = []
for i in range(len(X)):
    nn.forward_pass(X[i])
    outputs.append(nn.f[2])

outputs = np.array(outputs)

In [72]:
for i in range(12):
    print(nn.f[2][i], Y[i])

[[134.07710908]] [[123]]
[[137.02110238]] [[129]]
[[247.07820092]] [[243]]
[[169.68736529]] [[168]]
[[199.4096349]] [[201]]
[[176.42869636]] [[169]]
[[364.55170716]] [[357]]
[[82.45472542]] [[74]]
[[318.88001921]] [[322]]
[[249.56604173]] [[250]]
[[201.00023366]] [[202]]
[[705.82743956]] [[707]]


In [71]:
for i in range(12):
    nn.forward_pass()
    # print(nn.f[0])
    # print(nn.f[1])
    print(nn.f[2], Y[i])

[[[134.07710908]]

 [[137.02110238]]

 [[247.07820092]]

 [[169.68736529]]

 [[199.4096349 ]]

 [[176.42869636]]

 [[364.55170716]]

 [[ 82.45472542]]

 [[318.88001921]]

 [[249.56604173]]

 [[201.00023366]]

 [[705.82743956]]] [[123]]
[[[134.07710908]]

 [[137.02110238]]

 [[247.07820092]]

 [[169.68736529]]

 [[199.4096349 ]]

 [[176.42869636]]

 [[364.55170716]]

 [[ 82.45472542]]

 [[318.88001921]]

 [[249.56604173]]

 [[201.00023366]]

 [[705.82743956]]] [[129]]
[[[134.07710908]]

 [[137.02110238]]

 [[247.07820092]]

 [[169.68736529]]

 [[199.4096349 ]]

 [[176.42869636]]

 [[364.55170716]]

 [[ 82.45472542]]

 [[318.88001921]]

 [[249.56604173]]

 [[201.00023366]]

 [[705.82743956]]] [[243]]
[[[134.07710908]]

 [[137.02110238]]

 [[247.07820092]]

 [[169.68736529]]

 [[199.4096349 ]]

 [[176.42869636]]

 [[364.55170716]]

 [[ 82.45472542]]

 [[318.88001921]]

 [[249.56604173]]

 [[201.00023366]]

 [[705.82743956]]] [[168]]
[[[134.07710908]]

 [[137.02110238]]

 [[247.07820092]]


In [ ]:
# W1 = np.array([[2, 1, 5], [1, 4, 0], [3, 2, 1], [1, 3, 2]])
# B1 = np.array([1, 1, 0, 1])

# W2 = np.array([[1, 2, 3, 4]])
# B2 = np.array([2])
# W1.shape, W2.shape, B1.shape, B2.shape

The truth is there is no proof of convergence for deep neural networks because backpropogation doesn't find the global minimum and is often stuck in a local minima. How would we arrange these matrices in such a way that it would find the global minima is still quite a mystery to me.

It is just super easy to get stuck in a local minima and you might not ever be able to get out of that local minima because you are modifying all of those weights at the same time.